In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# ==========================================
# 1. LOAD DATASET KOMDIGI
# ==========================================
data_candidates = [
    Path("data/komdigi_hoaks.csv"),
    Path("../../data/komdigi_hoaks.csv"),
    Path("../../../data/komdigi_hoaks.csv"),
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("File data/komdigi_hoaks.csv tidak ditemukan.")

data = pd.read_csv(DATA_PATH, usecols=["title", "body_text", "topics"])
data["text"] = data["title"].fillna("") + " " + data["body_text"].fillna("")
data["topics"] = data["topics"].fillna("").str.strip()
data = data.loc[data["text"].str.strip().ne("") & data["topics"].ne("")].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data["topics"])

# Split berstratifikasi agar distribusi tiap topik terjaga.
X_train_val_text, X_test_text, y_train_val, y_test = train_test_split(
    data["text"], y, test_size=0.2, random_state=42, stratify=y
)
X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_train_val_text,
    y_train_val,
    test_size=0.125,
    random_state=42,
    stratify=y_train_val
)

# TF-IDF hanya di-fit pada data latih untuk mencegah kebocoran data.
vectorizer = TfidfVectorizer(max_features=512, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
X_val = vectorizer.transform(X_val_text).toarray().astype(np.float32)
X_test = vectorizer.transform(X_test_text).toarray().astype(np.float32)

class_ids = np.arange(len(label_encoder.classes_))
class_weights_values = compute_class_weight(
    class_weight="balanced", classes=class_ids, y=y_train
)
class_weights = dict(zip(class_ids, class_weights_values))

print("Jumlah data:", len(data))
print("Training:", len(X_train), "| Validasi:", len(X_val), "| Testing:", len(X_test))
print("Jumlah fitur TF-IDF:", X_train.shape[1])
print("Kelas topik:", list(label_encoder.classes_))

# ==========================================
# 2. BANGUN DAN LATIH MODEL JST
# ==========================================
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(label_encoder.classes_), activation="softmax")
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )
    ],
    verbose=1
)

# ==========================================
# 3. EVALUASI DAN PREDIKSI TOPIK
# ==========================================
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"Macro-F1      : {macro_f1:.4f}")
print("\nLaporan klasifikasi:")
print(classification_report(
    y_test,
    y_pred,
    labels=class_ids,
    target_names=label_encoder.classes_,
    zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

sample_index = 0
print("\nContoh prediksi artikel:")
print("Judul:", X_test_text.iloc[sample_index][:200])
print("Topik aktual:", label_encoder.inverse_transform([y_test[sample_index]])[0])
print("Topik prediksi:", label_encoder.inverse_transform([y_pred[sample_index]])[0])
